In [89]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR

WindowsPath('c:/Projects/creditlens-ai/data/raw')

In [90]:
bureau = pd.read_csv(
    DATA_DIR / "bureau.csv",
    low_memory=False
)

print(f"Rows: {bureau.shape[0]:,}")
print(f"Columns: {bureau.shape[1]}")
print(
    f"Memory usage: "
    f"{bureau.memory_usage(deep=True).sum() / (1024 ** 2):.2f} MB"
)

bureau.head()

Rows: 1,716,428
Columns: 17
Memory usage: 472.82 MB


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [91]:
bureau_customer_summary = pd.Series({
    "bureau_rows": len(bureau),
    "unique_customers": bureau["SK_ID_CURR"].nunique(),
    "unique_bureau_loans": bureau["SK_ID_BUREAU"].nunique(),
    "avg_loans_per_customer": (
        len(bureau) / bureau["SK_ID_CURR"].nunique()
    )
})

bureau_customer_summary

bureau_rows               1.716428e+06
unique_customers          3.058110e+05
unique_bureau_loans       1.716428e+06
avg_loans_per_customer    5.612709e+00
dtype: float64

In [92]:
bureau["CREDIT_ACTIVE"].value_counts(
    dropna=False
)

CREDIT_ACTIVE
Closed      1079273
Active       630607
Sold           6527
Bad debt         21
Name: count, dtype: int64

In [93]:
bureau_numeric_features = (
    bureau
    .groupby("SK_ID_CURR")
    .agg(
        BUREAU_LOAN_COUNT=("SK_ID_BUREAU", "count"),

        BUREAU_DAYS_CREDIT_MEAN=("DAYS_CREDIT", "mean"),
        BUREAU_DAYS_CREDIT_MIN=("DAYS_CREDIT", "min"),
        BUREAU_DAYS_CREDIT_MAX=("DAYS_CREDIT", "max"),

        BUREAU_CREDIT_DAY_OVERDUE_MEAN=(
            "CREDIT_DAY_OVERDUE",
            "mean"
        ),
        BUREAU_CREDIT_DAY_OVERDUE_MAX=(
            "CREDIT_DAY_OVERDUE",
            "max"
        ),

        BUREAU_CREDIT_SUM_MEAN=(
            "AMT_CREDIT_SUM",
            "mean"
        ),
        BUREAU_CREDIT_SUM_SUM=(
            "AMT_CREDIT_SUM",
            "sum"
        ),
        BUREAU_CREDIT_SUM_MAX=(
            "AMT_CREDIT_SUM",
            "max"
        ),

        BUREAU_DEBT_MEAN=(
            "AMT_CREDIT_SUM_DEBT",
            "mean"
        ),
        BUREAU_DEBT_SUM=(
            "AMT_CREDIT_SUM_DEBT",
            "sum"
        ),
        BUREAU_DEBT_MAX=(
            "AMT_CREDIT_SUM_DEBT",
            "max"
        ),

        BUREAU_OVERDUE_SUM=(
            "AMT_CREDIT_SUM_OVERDUE",
            "sum"
        ),
        BUREAU_OVERDUE_MAX=(
            "AMT_CREDIT_SUM_OVERDUE",
            "max"
        ),

        BUREAU_CREDIT_PROLONG_SUM=(
            "CNT_CREDIT_PROLONG",
            "sum"
        )
    )
    .reset_index()
)

bureau_numeric_features.head()

,SK_ID_CURR,BUREAU_LOAN_COUNT,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_MIN,BUREAU_DAYS_CREDIT_MAX,BUREAU_CREDIT_DAY_OVERDUE_MEAN,BUREAU_CREDIT_DAY_OVERDUE_MAX,BUREAU_CREDIT_SUM_MEAN,BUREAU_CREDIT_SUM_SUM,BUREAU_CREDIT_SUM_MAX,BUREAU_DEBT_MEAN,BUREAU_DEBT_SUM,BUREAU_DEBT_MAX,BUREAU_OVERDUE_SUM,BUREAU_OVERDUE_MAX,BUREAU_CREDIT_PROLONG_SUM
0,100001,7,-735.000000,-1572,-49,0.0,0,207623.571429,1453365.000,378000.0,85240.928571,596686.5,373239.0,0.0,0.0,0
1,100002,8,-874.000000,-1437,-103,0.0,0,108131.945625,865055.565,450000.0,49156.200000,245781.0,245781.0,0.0,0.0,0
2,100003,4,-1400.750000,-2586,-606,0.0,0,254350.125000,1017400.500,810000.0,0.000000,0.0,0.0,0.0,0.0,0
3,100004,2,-867.000000,-1326,-408,0.0,0,94518.900000,189037.800,94537.8,0.000000,0.0,0.0,0.0,0.0,0
4,100005,3,-190.666667,-373,-62,0.0,0,219042.000000,657126.000,568800.0,189469.500000,568408.5,543087.0,0.0,0.0,0


In [94]:
bureau_numeric_features = (
    bureau
    .groupby("SK_ID_CURR")
    .agg(
        BUREAU_LOAN_COUNT=("SK_ID_BUREAU", "count"),

        BUREAU_DAYS_CREDIT_MEAN=("DAYS_CREDIT", "mean"),
        BUREAU_DAYS_CREDIT_MIN=("DAYS_CREDIT", "min"),
        BUREAU_DAYS_CREDIT_MAX=("DAYS_CREDIT", "max"),

        BUREAU_CREDIT_DAY_OVERDUE_MEAN=(
            "CREDIT_DAY_OVERDUE",
            "mean"
        ),
        BUREAU_CREDIT_DAY_OVERDUE_MAX=(
            "CREDIT_DAY_OVERDUE",
            "max"
        ),

        BUREAU_CREDIT_SUM_MEAN=(
            "AMT_CREDIT_SUM",
            "mean"
        ),
        BUREAU_CREDIT_SUM_SUM=(
            "AMT_CREDIT_SUM",
            "sum"
        ),
        BUREAU_CREDIT_SUM_MAX=(
            "AMT_CREDIT_SUM",
            "max"
        ),

        BUREAU_DEBT_MEAN=(
            "AMT_CREDIT_SUM_DEBT",
            "mean"
        ),
        BUREAU_DEBT_SUM=(
            "AMT_CREDIT_SUM_DEBT",
            "sum"
        ),
        BUREAU_DEBT_MAX=(
            "AMT_CREDIT_SUM_DEBT",
            "max"
        ),

        BUREAU_OVERDUE_SUM=(
            "AMT_CREDIT_SUM_OVERDUE",
            "sum"
        ),
        BUREAU_OVERDUE_MAX=(
            "AMT_CREDIT_SUM_OVERDUE",
            "max"
        ),

        BUREAU_CREDIT_PROLONG_SUM=(
            "CNT_CREDIT_PROLONG",
            "sum"
        )
    )
    .reset_index()
)

bureau_numeric_features.head()

,SK_ID_CURR,BUREAU_LOAN_COUNT,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_MIN,BUREAU_DAYS_CREDIT_MAX,BUREAU_CREDIT_DAY_OVERDUE_MEAN,BUREAU_CREDIT_DAY_OVERDUE_MAX,BUREAU_CREDIT_SUM_MEAN,BUREAU_CREDIT_SUM_SUM,BUREAU_CREDIT_SUM_MAX,BUREAU_DEBT_MEAN,BUREAU_DEBT_SUM,BUREAU_DEBT_MAX,BUREAU_OVERDUE_SUM,BUREAU_OVERDUE_MAX,BUREAU_CREDIT_PROLONG_SUM
0,100001,7,-735.000000,-1572,-49,0.0,0,207623.571429,1453365.000,378000.0,85240.928571,596686.5,373239.0,0.0,0.0,0
1,100002,8,-874.000000,-1437,-103,0.0,0,108131.945625,865055.565,450000.0,49156.200000,245781.0,245781.0,0.0,0.0,0
2,100003,4,-1400.750000,-2586,-606,0.0,0,254350.125000,1017400.500,810000.0,0.000000,0.0,0.0,0.0,0.0,0
3,100004,2,-867.000000,-1326,-408,0.0,0,94518.900000,189037.800,94537.8,0.000000,0.0,0.0,0.0,0.0,0
4,100005,3,-190.666667,-373,-62,0.0,0,219042.000000,657126.000,568800.0,189469.500000,568408.5,543087.0,0.0,0.0,0


In [95]:
credit_status_counts = (
    pd.crosstab(
        bureau["SK_ID_CURR"],
        bureau["CREDIT_ACTIVE"]
    )
    .add_prefix("BUREAU_STATUS_")
    .reset_index()
)

credit_status_counts.head()

CREDIT_ACTIVE,SK_ID_CURR,BUREAU_STATUS_Active,BUREAU_STATUS_Bad debt,BUREAU_STATUS_Closed,BUREAU_STATUS_Sold
0,100001,3,0,4,0
1,100002,2,0,6,0
2,100003,1,0,3,0
3,100004,0,0,2,0
4,100005,2,0,1,0


In [96]:
bureau_features = bureau_numeric_features.merge(
    credit_status_counts,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

print(f"Rows: {bureau_features.shape[0]:,}")
print(f"Columns: {bureau_features.shape[1]}")

bureau_features.head()

Rows: 305,811
Columns: 20


,SK_ID_CURR,BUREAU_LOAN_COUNT,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_MIN,BUREAU_DAYS_CREDIT_MAX,BUREAU_CREDIT_DAY_OVERDUE_MEAN,BUREAU_CREDIT_DAY_OVERDUE_MAX,BUREAU_CREDIT_SUM_MEAN,BUREAU_CREDIT_SUM_SUM,BUREAU_CREDIT_SUM_MAX,BUREAU_DEBT_MEAN,BUREAU_DEBT_SUM,BUREAU_DEBT_MAX,BUREAU_OVERDUE_SUM,BUREAU_OVERDUE_MAX,BUREAU_CREDIT_PROLONG_SUM,BUREAU_STATUS_Active,BUREAU_STATUS_Bad debt,BUREAU_STATUS_Closed,BUREAU_STATUS_Sold
0,100001,7,-735.000000,-1572,-49,0.0,0,207623.571429,1453365.000,378000.0,85240.928571,596686.5,373239.0,0.0,0.0,0,3,0,4,0
1,100002,8,-874.000000,-1437,-103,0.0,0,108131.945625,865055.565,450000.0,49156.200000,245781.0,245781.0,0.0,0.0,0,2,0,6,0
2,100003,4,-1400.750000,-2586,-606,0.0,0,254350.125000,1017400.500,810000.0,0.000000,0.0,0.0,0.0,0.0,0,1,0,3,0
3,100004,2,-867.000000,-1326,-408,0.0,0,94518.900000,189037.800,94537.8,0.000000,0.0,0.0,0.0,0.0,0,0,0,2,0
4,100005,3,-190.666667,-373,-62,0.0,0,219042.000000,657126.000,568800.0,189469.500000,568408.5,543087.0,0.0,0.0,0,2,0,1,0


In [97]:
print(
    "Unique customers:",
    bureau_features["SK_ID_CURR"].nunique()
)

print(
    "SK_ID_CURR unique:",
    bureau_features["SK_ID_CURR"].is_unique
)

Unique customers: 305811
SK_ID_CURR unique: True


In [98]:
bureau_features_path = (
    INTERIM_DIR / "bureau_customer_features.csv"
)

bureau_features.to_csv(
    bureau_features_path,
    index=False
)

print(f"Saved to: {bureau_features_path}")

Saved to: c:\Projects\creditlens-ai\data\interim\bureau_customer_features.csv


In [99]:
import gc

del bureau
del bureau_numeric_features
del credit_status_counts

gc.collect()

print("Raw bureau objects removed from memory.")

Raw bureau objects removed from memory.


In [100]:
application_train_ids = pd.read_csv(
    DATA_DIR / "application_train.csv",
    usecols=["SK_ID_CURR", "TARGET"]
)

bureau_coverage = application_train_ids.merge(
    bureau_features[["SK_ID_CURR"]],
    on="SK_ID_CURR",
    how="left",
    indicator=True,
    validate="one_to_one"
)

coverage_summary = (
    bureau_coverage["_merge"]
    .value_counts()
)

coverage_summary

_merge
both          263491
left_only      44020
right_only         0
Name: count, dtype: int64

In [101]:
matched_customers = (
    bureau_coverage["_merge"] == "both"
).sum()

total_customers = len(bureau_coverage)

print(
    f"Customers with bureau history: "
    f"{matched_customers:,}"
)

print(
    f"Coverage: "
    f"{matched_customers / total_customers * 100:.2f}%"
)

Customers with bureau history: 263,491
Coverage: 85.69%


In [102]:
application_train = pd.read_csv(
    DATA_DIR / "application_train.csv",
    low_memory=False
)

train_with_bureau = application_train.merge(
    bureau_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

print(
    f"Original shape: "
    f"{application_train.shape}"
)

print(
    f"With bureau shape: "
    f"{train_with_bureau.shape}"
)

Original shape: (307511, 122)
With bureau shape: (307511, 141)


In [103]:
train_with_bureau["BUREAU_HAS_HISTORY"] = (
    train_with_bureau["BUREAU_LOAN_COUNT"]
    .notna()
    .astype("int8")
)

count_columns = [
    "BUREAU_LOAN_COUNT",
    "BUREAU_STATUS_Active",
    "BUREAU_STATUS_Bad debt",
    "BUREAU_STATUS_Closed",
    "BUREAU_STATUS_Sold"
]

train_with_bureau[count_columns] = (
    train_with_bureau[count_columns]
    .fillna(0)
)

train_with_bureau[
    [
        "SK_ID_CURR",
        "BUREAU_HAS_HISTORY",
        *count_columns
    ]
].head(10)

C:\Users\Vıctus\AppData\Local\Temp\ipykernel_23348\1847904526.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_with_bureau["BUREAU_HAS_HISTORY"] = (


,SK_ID_CURR,BUREAU_HAS_HISTORY,BUREAU_LOAN_COUNT,BUREAU_STATUS_Active,BUREAU_STATUS_Bad debt,BUREAU_STATUS_Closed,BUREAU_STATUS_Sold
0,100002,1,8.0,2.0,0.0,6.0,0.0
1,100003,1,4.0,1.0,0.0,3.0,0.0
2,100004,1,2.0,0.0,0.0,2.0,0.0
3,100006,0,0.0,0.0,0.0,0.0,0.0
4,100007,1,1.0,0.0,0.0,1.0,0.0
5,100008,1,3.0,1.0,0.0,2.0,0.0
6,100009,1,18.0,4.0,0.0,14.0,0.0
7,100010,1,2.0,1.0,0.0,1.0,0.0
8,100011,1,4.0,0.0,0.0,4.0,0.0
9,100012,0,0.0,0.0,0.0,0.0,0.0


In [104]:
# Fragmentation uyarısını temizlemek ve bellekte
# daha düzenli bir DataFrame oluşturmak için.
train_with_bureau = train_with_bureau.copy()

print(f"Final train + bureau shape: {train_with_bureau.shape}")

Final train + bureau shape: (307511, 142)


In [105]:
del application_train
del application_train_ids
del bureau_coverage

gc.collect()

print("Unused intermediate objects removed.")

Unused intermediate objects removed.


In [106]:
y_bureau = (
    train_with_bureau["TARGET"]
    .astype("int8")
)

X_bureau = (
    train_with_bureau
    .drop(
        columns=[
            "TARGET",
            "SK_ID_CURR"
        ]
    )
    .copy()
)

X_bureau["DAYS_EMPLOYED_ANOMALY"] = (
    X_bureau["DAYS_EMPLOYED"] == 365243
).astype("int8")

X_bureau.loc[
    X_bureau["DAYS_EMPLOYED"] == 365243,
    "DAYS_EMPLOYED"
] = np.nan

print(f"X shape: {X_bureau.shape}")
print(f"y shape: {y_bureau.shape}")

X shape: (307511, 141)
y shape: (307511,)


In [107]:
from sklearn.model_selection import train_test_split

X_train_bureau, X_val_bureau, y_train_bureau, y_val_bureau = (
    train_test_split(
        X_bureau,
        y_bureau,
        test_size=0.20,
        random_state=42,
        stratify=y_bureau
    )
)

print(f"Train: {X_train_bureau.shape}")
print(f"Validation: {X_val_bureau.shape}")

print(
    f"Train positive rate: "
    f"{y_train_bureau.mean() * 100:.2f}%"
)

print(
    f"Validation positive rate: "
    f"{y_val_bureau.mean() * 100:.2f}%"
)

Train: (246008, 141)
Validation: (61503, 141)
Train positive rate: 8.07%
Validation positive rate: 8.07%


In [108]:
numeric_columns_bureau = (
    X_train_bureau
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

categorical_columns_bureau = (
    X_train_bureau
    .select_dtypes(exclude=["number"])
    .columns
    .tolist()
)

print(
    f"Numeric features: "
    f"{len(numeric_columns_bureau)}"
)

print(
    f"Categorical features: "
    f"{len(categorical_columns_bureau)}"
)

print(
    f"Total: "
    f"{len(numeric_columns_bureau) + len(categorical_columns_bureau)}"
)

Numeric features: 125
Categorical features: 16
Total: 141


In [109]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

In [110]:
numeric_pipeline_bureau = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline_bureau = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

preprocessor_bureau = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline_bureau,
            numeric_columns_bureau
        ),
        (
            "categorical",
            categorical_pipeline_bureau,
            categorical_columns_bureau
        )
    ],
    sparse_threshold=1.0
)

In [111]:
import time

bureau_balanced_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_bureau
        ),
        (
            "classifier",
            LogisticRegression(
                solver="saga",
                max_iter=1500,
                tol=1e-3,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

start_time = time.time()

bureau_balanced_model.fit(
    X_train_bureau,
    y_train_bureau
)

bureau_training_time = (
    time.time() - start_time
)

bureau_classifier = (
    bureau_balanced_model
    .named_steps["classifier"]
)

print(
    f"Training time: "
    f"{bureau_training_time:.2f} seconds"
)

print(
    f"Iterations used: "
    f"{bureau_classifier.n_iter_[0]}"
)

Training time: 123.37 seconds
Iterations used: 369


In [112]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

In [113]:
bureau_probabilities = (
    bureau_balanced_model.predict_proba(
        X_val_bureau
    )[:, 1]
)

bureau_predictions = (
    bureau_probabilities >= 0.50
).astype(int)

bureau_metrics = {
    "accuracy": accuracy_score(
        y_val_bureau,
        bureau_predictions
    ),
    "precision": precision_score(
        y_val_bureau,
        bureau_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_val_bureau,
        bureau_predictions,
        zero_division=0
    ),
    "f1": f1_score(
        y_val_bureau,
        bureau_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_val_bureau,
        bureau_probabilities
    ),
    "pr_auc": average_precision_score(
        y_val_bureau,
        bureau_probabilities
    )
}

pd.Series(bureau_metrics)

accuracy     0.691348
precision    0.163353
recall       0.684995
f1           0.263797
roc_auc      0.753412
pr_auc       0.235836
dtype: float64

In [114]:
bureau_confusion_matrix = confusion_matrix(
    y_val_bureau,
    bureau_predictions
)

bureau_confusion_matrix

array([[39119, 17419],
       [ 1564,  3401]])

In [115]:
baseline_balanced_metrics = {
    "accuracy": 0.689300,
    "precision": 0.161368,
    "recall": 0.678751,
    "f1": 0.260745,
    "roc_auc": 0.748826,
    "pr_auc": 0.228761
}

bureau_comparison = pd.DataFrame({
    "Application_Only": baseline_balanced_metrics,
    "Application_Plus_Bureau": bureau_metrics
}).T

bureau_comparison

,accuracy,precision,recall,f1,roc_auc,pr_auc
Application_Only,0.689300,0.161368,0.678751,0.260745,0.748826,0.228761
Application_Plus_Bureau,0.691348,0.163353,0.684995,0.263797,0.753412,0.235836


In [116]:
metric_improvement = (
    bureau_comparison
    .loc["Application_Plus_Bureau"]
    -
    bureau_comparison
    .loc["Application_Only"]
)

metric_improvement

accuracy     0.002048
precision    0.001985
recall       0.006244
f1           0.003052
roc_auc      0.004586
pr_auc       0.007075
dtype: float64

In [117]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

bureau_comparison_path = (
    REPORTS_DIR / "bureau_feature_experiment.csv"
)

bureau_comparison.to_csv(
    bureau_comparison_path,
    index=True
)

print(f"Saved to: {bureau_comparison_path}")

Saved to: c:\Projects\creditlens-ai\reports\bureau_feature_experiment.csv


In [118]:
bureau_confusion_results = pd.DataFrame({
    "model": [
        "Application_Only_Balanced",
        "Application_Plus_Bureau_Balanced"
    ],
    "TN": [
        39024,
        bureau_confusion_matrix[0, 0]
    ],
    "FP": [
        17514,
        bureau_confusion_matrix[0, 1]
    ],
    "FN": [
        1595,
        bureau_confusion_matrix[1, 0]
    ],
    "TP": [
        3370,
        bureau_confusion_matrix[1, 1]
    ]
})

bureau_confusion_results.to_csv(
    REPORTS_DIR / "bureau_confusion_comparison.csv",
    index=False
)

bureau_confusion_results

,model,TN,FP,FN,TP
0,Application_Only_Balanced,39024,17514,1595,3370
1,Application_Plus_Bureau_Balanced,39119,17419,1564,3401


## Bureau Feature Engineering Result

Customer-level historical credit features were engineered from approximately 1.7 million bureau records and joined to the application dataset.

Bureau information was available for 85.69% of training applicants.

Adding bureau-derived features improved the balanced Logistic Regression validation performance:

- ROC-AUC: 0.7488 → 0.7534
- PR-AUC: 0.2288 → 0.2358
- Recall: 0.6788 → 0.6850
- F1: 0.2607 → 0.2638

The experiment indicates that historical credit behavior contains additional predictive information beyond the application-level features.

In [119]:
del X_bureau
del X_train_bureau
del X_val_bureau
del y_bureau
del y_train_bureau
del y_val_bureau
del train_with_bureau

gc.collect()

print("Bureau experiment objects removed from memory.")

Bureau experiment objects removed from memory.


In [120]:
previous = pd.read_csv(
    DATA_DIR / "previous_application.csv",
    low_memory=False
)

print(f"Rows: {previous.shape[0]:,}")
print(f"Columns: {previous.shape[1]}")
print(
    f"Memory usage: "
    f"{previous.memory_usage(deep=True).sum() / (1024 ** 2):.2f} MB"
)

previous.head()

Rows: 1,670,214
Columns: 37
Memory usage: 1703.01 MB


,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [121]:
previous_summary = pd.Series({
    "rows": len(previous),
    "unique_customers": previous["SK_ID_CURR"].nunique(),
    "unique_previous_applications": previous["SK_ID_PREV"].nunique(),
    "avg_applications_per_customer": (
        len(previous)
        / previous["SK_ID_CURR"].nunique()
    )
})

previous_summary

rows                             1.670214e+06
unique_customers                 3.388570e+05
unique_previous_applications     1.670214e+06
avg_applications_per_customer    4.928964e+00
dtype: float64

In [122]:
previous["NAME_CONTRACT_STATUS"].value_counts(
    dropna=False
)

NAME_CONTRACT_STATUS
Approved        1036781
Canceled         316319
Refused          290678
Unused offer      26436
Name: count, dtype: int64

In [123]:
previous[
    [
        "AMT_APPLICATION",
        "AMT_CREDIT",
        "AMT_ANNUITY",
        "AMT_DOWN_PAYMENT"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
AMT_APPLICATION,1670214.0,175233.860360,292779.762386,0.0,18720.00,71046.0,180360.00,6905160.000
AMT_CREDIT,1670213.0,196114.021218,318574.616547,0.0,24160.50,80541.0,216418.50,6905160.000
AMT_ANNUITY,1297979.0,15955.120659,14782.137335,0.0,6321.78,11250.0,20658.42,418058.145
AMT_DOWN_PAYMENT,774370.0,6697.402139,20921.495410,-0.9,0.00,1638.0,7740.00,3060045.000


In [124]:
previous_numeric_features = (
    previous
    .groupby("SK_ID_CURR")
    .agg(
        PREV_APPLICATION_COUNT=(
            "SK_ID_PREV",
            "count"
        ),

        PREV_AMT_APPLICATION_MEAN=(
            "AMT_APPLICATION",
            "mean"
        ),
        PREV_AMT_APPLICATION_SUM=(
            "AMT_APPLICATION",
            "sum"
        ),
        PREV_AMT_APPLICATION_MAX=(
            "AMT_APPLICATION",
            "max"
        ),

        PREV_AMT_CREDIT_MEAN=(
            "AMT_CREDIT",
            "mean"
        ),
        PREV_AMT_CREDIT_SUM=(
            "AMT_CREDIT",
            "sum"
        ),
        PREV_AMT_CREDIT_MAX=(
            "AMT_CREDIT",
            "max"
        ),

        PREV_AMT_ANNUITY_MEAN=(
            "AMT_ANNUITY",
            "mean"
        ),
        PREV_AMT_ANNUITY_MAX=(
            "AMT_ANNUITY",
            "max"
        ),

        PREV_DOWN_PAYMENT_MEAN=(
            "AMT_DOWN_PAYMENT",
            "mean"
        ),

        PREV_CNT_PAYMENT_MEAN=(
            "CNT_PAYMENT",
            "mean"
        ),
        PREV_CNT_PAYMENT_MAX=(
            "CNT_PAYMENT",
            "max"
        ),

        PREV_DAYS_DECISION_MEAN=(
            "DAYS_DECISION",
            "mean"
        ),
        PREV_DAYS_DECISION_MIN=(
            "DAYS_DECISION",
            "min"
        ),
        PREV_DAYS_DECISION_MAX=(
            "DAYS_DECISION",
            "max"
        )
    )
    .reset_index()
)

previous_numeric_features.head()

,SK_ID_CURR,PREV_APPLICATION_COUNT,PREV_AMT_APPLICATION_MEAN,PREV_AMT_APPLICATION_SUM,PREV_AMT_APPLICATION_MAX,PREV_AMT_CREDIT_MEAN,PREV_AMT_CREDIT_SUM,PREV_AMT_CREDIT_MAX,PREV_AMT_ANNUITY_MEAN,PREV_AMT_ANNUITY_MAX,PREV_DOWN_PAYMENT_MEAN,PREV_CNT_PAYMENT_MEAN,PREV_CNT_PAYMENT_MAX,PREV_DAYS_DECISION_MEAN,PREV_DAYS_DECISION_MIN,PREV_DAYS_DECISION_MAX
0,100001,1,24835.50,24835.5,24835.5,23787.00,23787.0,23787.0,3951.000,3951.000,2520.0,8.0,8.0,-1740.0,-1740,-1740
1,100002,1,179055.00,179055.0,179055.0,179055.00,179055.0,179055.0,9251.775,9251.775,0.0,24.0,24.0,-606.0,-606,-606
2,100003,3,435436.50,1306309.5,900000.0,484191.00,1452573.0,1035882.0,56553.990,98356.995,3442.5,10.0,12.0,-1305.0,-2341,-746
3,100004,1,24282.00,24282.0,24282.0,20106.00,20106.0,20106.0,5357.250,5357.250,4860.0,4.0,4.0,-815.0,-815,-815
4,100005,2,22308.75,44617.5,44617.5,20076.75,40153.5,40153.5,4813.200,4813.200,4464.0,12.0,12.0,-536.0,-757,-315


In [125]:
previous_status_counts = (
    pd.crosstab(
        previous["SK_ID_CURR"],
        previous["NAME_CONTRACT_STATUS"]
    )
    .add_prefix("PREV_STATUS_")
    .reset_index()
)

previous_status_counts.head()

NAME_CONTRACT_STATUS,SK_ID_CURR,PREV_STATUS_Approved,PREV_STATUS_Canceled,PREV_STATUS_Refused,PREV_STATUS_Unused offer
0,100001,1,0,0,0
1,100002,1,0,0,0
2,100003,3,0,0,0
3,100004,1,0,0,0
4,100005,1,1,0,0


In [126]:
previous_features = (
    previous_numeric_features
    .merge(
        previous_status_counts,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
)

previous_features.head()

,SK_ID_CURR,PREV_APPLICATION_COUNT,PREV_AMT_APPLICATION_MEAN,PREV_AMT_APPLICATION_SUM,PREV_AMT_APPLICATION_MAX,PREV_AMT_CREDIT_MEAN,PREV_AMT_CREDIT_SUM,PREV_AMT_CREDIT_MAX,PREV_AMT_ANNUITY_MEAN,PREV_AMT_ANNUITY_MAX,PREV_DOWN_PAYMENT_MEAN,PREV_CNT_PAYMENT_MEAN,PREV_CNT_PAYMENT_MAX,PREV_DAYS_DECISION_MEAN,PREV_DAYS_DECISION_MIN,PREV_DAYS_DECISION_MAX,PREV_STATUS_Approved,PREV_STATUS_Canceled,PREV_STATUS_Refused,PREV_STATUS_Unused offer
0,100001,1,24835.50,24835.5,24835.5,23787.00,23787.0,23787.0,3951.000,3951.000,2520.0,8.0,8.0,-1740.0,-1740,-1740,1,0,0,0
1,100002,1,179055.00,179055.0,179055.0,179055.00,179055.0,179055.0,9251.775,9251.775,0.0,24.0,24.0,-606.0,-606,-606,1,0,0,0
2,100003,3,435436.50,1306309.5,900000.0,484191.00,1452573.0,1035882.0,56553.990,98356.995,3442.5,10.0,12.0,-1305.0,-2341,-746,3,0,0,0
3,100004,1,24282.00,24282.0,24282.0,20106.00,20106.0,20106.0,5357.250,5357.250,4860.0,4.0,4.0,-815.0,-815,-815,1,0,0,0
4,100005,2,22308.75,44617.5,44617.5,20076.75,40153.5,40153.5,4813.200,4813.200,4464.0,12.0,12.0,-536.0,-757,-315,1,1,0,0


In [127]:
previous_features["PREV_APPROVAL_RATE"] = (
    previous_features["PREV_STATUS_Approved"]
    / previous_features["PREV_APPLICATION_COUNT"]
)

previous_features["PREV_REFUSAL_RATE"] = (
    previous_features["PREV_STATUS_Refused"]
    / previous_features["PREV_APPLICATION_COUNT"]
)

previous_features["PREV_CANCELLATION_RATE"] = (
    previous_features["PREV_STATUS_Canceled"]
    / previous_features["PREV_APPLICATION_COUNT"]
)

previous_features[
    [
        "SK_ID_CURR",
        "PREV_APPLICATION_COUNT",
        "PREV_STATUS_Approved",
        "PREV_STATUS_Refused",
        "PREV_APPROVAL_RATE",
        "PREV_REFUSAL_RATE"
    ]
].head(10)

,SK_ID_CURR,PREV_APPLICATION_COUNT,PREV_STATUS_Approved,PREV_STATUS_Refused,PREV_APPROVAL_RATE,PREV_REFUSAL_RATE
0,100001,1,1,0,1.000000,0.000000
1,100002,1,1,0,1.000000,0.000000
2,100003,3,3,0,1.000000,0.000000
3,100004,1,1,0,1.000000,0.000000
4,100005,2,1,0,0.500000,0.000000
5,100006,9,5,1,0.555556,0.111111
6,100007,6,6,0,1.000000,0.000000
7,100008,5,4,0,0.800000,0.000000
8,100009,7,7,0,1.000000,0.000000
9,100010,1,1,0,1.000000,0.000000


In [128]:
previous_features["PREV_CREDIT_TO_APPLICATION_RATIO"] = (
    previous_features["PREV_AMT_CREDIT_SUM"]
    / previous_features["PREV_AMT_APPLICATION_SUM"]
    .replace(0, np.nan)
)

previous_features[
    [
        "SK_ID_CURR",
        "PREV_AMT_APPLICATION_SUM",
        "PREV_AMT_CREDIT_SUM",
        "PREV_CREDIT_TO_APPLICATION_RATIO"
    ]
].head()

,SK_ID_CURR,PREV_AMT_APPLICATION_SUM,PREV_AMT_CREDIT_SUM,PREV_CREDIT_TO_APPLICATION_RATIO
0,100001,24835.5,23787.0,0.957782
1,100002,179055.0,179055.0,1.000000
2,100003,1306309.5,1452573.0,1.111967
3,100004,24282.0,20106.0,0.828021
4,100005,44617.5,40153.5,0.899950


In [129]:
print(
    f"Rows: {previous_features.shape[0]:,}"
)

print(
    f"Columns: {previous_features.shape[1]}"
)

print(
    "Unique customers:",
    previous_features["SK_ID_CURR"].nunique()
)

print(
    "SK_ID_CURR unique:",
    previous_features["SK_ID_CURR"].is_unique
)

Rows: 338,857
Columns: 24
Unique customers: 338857
SK_ID_CURR unique: True


In [130]:
previous_features_path = (
    INTERIM_DIR / "previous_application_customer_features.csv"
)

previous_features.to_csv(
    previous_features_path,
    index=False
)

print(f"Saved to: {previous_features_path}")

Saved to: c:\Projects\creditlens-ai\data\interim\previous_application_customer_features.csv


In [131]:
del previous
del previous_numeric_features
del previous_status_counts

gc.collect()

print("Raw previous-application objects removed from memory.")

Raw previous-application objects removed from memory.


In [132]:
application_train = pd.read_csv(
    DATA_DIR / "application_train.csv",
    low_memory=False
)

print(f"Application shape: {application_train.shape}")

Application shape: (307511, 122)


In [133]:
previous_coverage = (
    application_train[["SK_ID_CURR"]]
    .merge(
        previous_features[["SK_ID_CURR"]],
        on="SK_ID_CURR",
        how="left",
        indicator=True,
        validate="one_to_one"
    )
)

previous_coverage["_merge"].value_counts()

_merge
both          291057
left_only      16454
right_only         0
Name: count, dtype: int64

In [134]:
previous_matched = (
    previous_coverage["_merge"] == "both"
).sum()

previous_total = len(previous_coverage)

print(
    f"Customers with previous application history: "
    f"{previous_matched:,}"
)

print(
    f"Coverage: "
    f"{previous_matched / previous_total * 100:.2f}%"
)

Customers with previous application history: 291,057
Coverage: 94.65%


In [135]:
train_relational = (
    application_train
    .merge(
        bureau_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
    .merge(
        previous_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
)

print(
    f"Relational train shape before history flags: "
    f"{train_relational.shape}"
)

Relational train shape before history flags: (307511, 164)


In [136]:
train_relational["BUREAU_HAS_HISTORY"] = (
    train_relational["BUREAU_LOAN_COUNT"]
    .notna()
    .astype("int8")
)

train_relational["PREV_HAS_HISTORY"] = (
    train_relational["PREV_APPLICATION_COUNT"]
    .notna()
    .astype("int8")
)

C:\Users\Vıctus\AppData\Local\Temp\ipykernel_23348\3945503393.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_relational["BUREAU_HAS_HISTORY"] = (
C:\Users\Vıctus\AppData\Local\Temp\ipykernel_23348\3945503393.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_relational["PREV_HAS_HISTORY"] = (


In [137]:
bureau_count_columns = [
    "BUREAU_LOAN_COUNT",
    "BUREAU_STATUS_Active",
    "BUREAU_STATUS_Bad debt",
    "BUREAU_STATUS_Closed",
    "BUREAU_STATUS_Sold"
]

previous_count_columns = [
    "PREV_APPLICATION_COUNT",
    "PREV_STATUS_Approved",
    "PREV_STATUS_Canceled",
    "PREV_STATUS_Refused",
    "PREV_STATUS_Unused offer"
]

train_relational[
    bureau_count_columns + previous_count_columns
] = (
    train_relational[
        bureau_count_columns + previous_count_columns
    ]
    .fillna(0)
)

In [138]:
train_relational = train_relational.copy()

print(
    f"Final relational train shape: "
    f"{train_relational.shape}"
)

Final relational train shape: (307511, 166)


In [139]:
y_relational = (
    train_relational["TARGET"]
    .astype("int8")
)

X_relational = (
    train_relational
    .drop(
        columns=[
            "TARGET",
            "SK_ID_CURR"
        ]
    )
    .copy()
)

X_relational["DAYS_EMPLOYED_ANOMALY"] = (
    X_relational["DAYS_EMPLOYED"] == 365243
).astype("int8")

X_relational.loc[
    X_relational["DAYS_EMPLOYED"] == 365243,
    "DAYS_EMPLOYED"
] = np.nan

print(f"X shape: {X_relational.shape}")
print(f"y shape: {y_relational.shape}")

X shape: (307511, 165)
y shape: (307511,)


In [140]:
print(
    "Duplicate customers:",
    train_relational["SK_ID_CURR"].duplicated().sum()
)

print(
    "Target missing:",
    train_relational["TARGET"].isna().sum()
)

print(
    "Bureau history rate:",
    f"{train_relational['BUREAU_HAS_HISTORY'].mean() * 100:.2f}%"
)

print(
    "Previous history rate:",
    f"{train_relational['PREV_HAS_HISTORY'].mean() * 100:.2f}%"
)

Duplicate customers: 0
Target missing: 0
Bureau history rate: 85.69%
Previous history rate: 94.65%


In [141]:
from sklearn.model_selection import train_test_split

X_train_rel, X_val_rel, y_train_rel, y_val_rel = train_test_split(
    X_relational,
    y_relational,
    test_size=0.20,
    random_state=42,
    stratify=y_relational
)

print(f"Train shape: {X_train_rel.shape}")
print(f"Validation shape: {X_val_rel.shape}")

print(
    f"Train positive rate: "
    f"{y_train_rel.mean() * 100:.2f}%"
)

print(
    f"Validation positive rate: "
    f"{y_val_rel.mean() * 100:.2f}%"
)

Train shape: (246008, 165)
Validation shape: (61503, 165)
Train positive rate: 8.07%
Validation positive rate: 8.07%


In [142]:
numeric_columns_rel = (
    X_train_rel
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

categorical_columns_rel = (
    X_train_rel
    .select_dtypes(exclude=["number"])
    .columns
    .tolist()
)

print(f"Numeric features: {len(numeric_columns_rel)}")
print(f"Categorical features: {len(categorical_columns_rel)}")
print(
    f"Total features: "
    f"{len(numeric_columns_rel) + len(categorical_columns_rel)}"
)

Numeric features: 149
Categorical features: 16
Total features: 165


In [143]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

numeric_pipeline_rel = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline_rel = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

preprocessor_rel = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline_rel,
            numeric_columns_rel
        ),
        (
            "categorical",
            categorical_pipeline_rel,
            categorical_columns_rel
        )
    ],
    sparse_threshold=1.0
)

preprocessor_rel

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",1.0
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fe

In [144]:
import time

relational_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_rel
        ),
        (
            "classifier",
            LogisticRegression(
                solver="saga",
                max_iter=1500,
                tol=1e-3,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

start_time = time.time()

relational_model.fit(
    X_train_rel,
    y_train_rel
)

relational_training_time = (
    time.time() - start_time
)

relational_classifier = (
    relational_model
    .named_steps["classifier"]
)

print(
    f"Training time: "
    f"{relational_training_time:.2f} seconds"
)

print(
    f"Iterations used: "
    f"{relational_classifier.n_iter_[0]}"
)

Training time: 142.12 seconds
Iterations used: 369


In [145]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

relational_probabilities = (
    relational_model.predict_proba(
        X_val_rel
    )[:, 1]
)

relational_predictions = (
    relational_probabilities >= 0.50
).astype(int)

relational_metrics = {
    "accuracy": accuracy_score(
        y_val_rel,
        relational_predictions
    ),
    "precision": precision_score(
        y_val_rel,
        relational_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_val_rel,
        relational_predictions,
        zero_division=0
    ),
    "f1": f1_score(
        y_val_rel,
        relational_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_val_rel,
        relational_probabilities
    ),
    "pr_auc": average_precision_score(
        y_val_rel,
        relational_probabilities
    )
}

pd.Series(relational_metrics)

accuracy     0.696307
precision    0.166982
recall       0.692447
f1           0.269077
roc_auc      0.760525
pr_auc       0.243105
dtype: float64

In [146]:
relational_confusion_matrix = confusion_matrix(
    y_val_rel,
    relational_predictions
)

relational_confusion_matrix

array([[39387, 17151],
       [ 1527,  3438]])

In [147]:
application_only_metrics = {
    "accuracy": 0.689300,
    "precision": 0.161368,
    "recall": 0.678751,
    "f1": 0.260745,
    "roc_auc": 0.748826,
    "pr_auc": 0.228761
}

application_bureau_metrics = {
    "accuracy": 0.691348,
    "precision": 0.163353,
    "recall": 0.684995,
    "f1": 0.263797,
    "roc_auc": 0.753412,
    "pr_auc": 0.235836
}

relational_comparison = pd.DataFrame({
    "Application_Only": application_only_metrics,
    "Application_Plus_Bureau": application_bureau_metrics,
    "Application_Plus_Bureau_Plus_Previous": relational_metrics
}).T

relational_comparison

,accuracy,precision,recall,f1,roc_auc,pr_auc
Application_Only,0.689300,0.161368,0.678751,0.260745,0.748826,0.228761
Application_Plus_Bureau,0.691348,0.163353,0.684995,0.263797,0.753412,0.235836
Application_Plus_Bureau_Plus_Previous,0.696307,0.166982,0.692447,0.269077,0.760525,0.243105


In [148]:
previous_incremental_improvement = (
    relational_comparison
    .loc["Application_Plus_Bureau_Plus_Previous"]
    -
    relational_comparison
    .loc["Application_Plus_Bureau"]
)

previous_incremental_improvement

accuracy     0.004959
precision    0.003629
recall       0.007452
f1           0.005280
roc_auc      0.007113
pr_auc       0.007269
dtype: float64

In [149]:
total_relational_improvement = (
    relational_comparison
    .loc["Application_Plus_Bureau_Plus_Previous"]
    -
    relational_comparison
    .loc["Application_Only"]
)

total_relational_improvement

accuracy     0.007007
precision    0.005614
recall       0.013696
f1           0.008332
roc_auc      0.011699
pr_auc       0.014344
dtype: float64

In [150]:
relational_comparison_path = (
    REPORTS_DIR / "relational_feature_experiment.csv"
)

relational_comparison.to_csv(
    relational_comparison_path,
    index=True
)

print(f"Saved to: {relational_comparison_path}")

Saved to: c:\Projects\creditlens-ai\reports\relational_feature_experiment.csv


In [151]:
relational_confusion_results = pd.DataFrame({
    "model": [
        "Application_Only_Balanced",
        "Application_Plus_Bureau_Balanced",
        "Application_Plus_Bureau_Plus_Previous_Balanced"
    ],
    "TN": [
        39024,
        39119,
        relational_confusion_matrix[0, 0]
    ],
    "FP": [
        17514,
        17419,
        relational_confusion_matrix[0, 1]
    ],
    "FN": [
        1595,
        1564,
        relational_confusion_matrix[1, 0]
    ],
    "TP": [
        3370,
        3401,
        relational_confusion_matrix[1, 1]
    ]
})

relational_confusion_results.to_csv(
    REPORTS_DIR / "relational_confusion_comparison.csv",
    index=False
)

relational_confusion_results

,model,TN,FP,FN,TP
0,Application_Only_Balanced,39024,17514,1595,3370
1,Application_Plus_Bureau_Balanced,39119,17419,1564,3401
2,Application_Plus_Bureau_Plus_Previous_Balanced,39387,17151,1527,3438


## Previous Application Feature Engineering Result

Customer-level behavioral features were engineered from approximately 1.67 million historical Home Credit applications.

Previous-application history was available for 94.65% of training applicants.

Adding previous-application features on top of application and bureau features improved validation performance:

- ROC-AUC: 0.7534 → 0.7605
- PR-AUC: 0.2358 → 0.2431
- Recall: 0.6850 → 0.6924
- F1: 0.2638 → 0.2691

Compared with the original application-only model:

- ROC-AUC: 0.7488 → 0.7605
- PR-AUC: 0.2288 → 0.2431

The results indicate that historical application behavior provides additional predictive information beyond both the current application and bureau credit-history features.

In [152]:
del relational_model
del relational_probabilities
del relational_predictions

del X_relational
del X_train_rel
del X_val_rel

del y_relational
del y_train_rel
del y_val_rel

del train_relational
del application_train
del previous_coverage

gc.collect()

print("Relational model objects removed from memory.")

Relational model objects removed from memory.


In [153]:
installments = pd.read_csv(
    DATA_DIR / "installments_payments.csv",
    low_memory=False
)

print(f"Rows: {installments.shape[0]:,}")
print(f"Columns: {installments.shape[1]}")

print(
    f"Memory usage: "
    f"{installments.memory_usage(deep=True).sum() / (1024 ** 2):.2f} MB"
)

installments.head()

Rows: 13,605,401
Columns: 8
Memory usage: 830.41 MB


,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585


In [154]:
installments_summary = pd.Series({
    "rows": len(installments),
    "unique_customers": installments["SK_ID_CURR"].nunique(),
    "unique_previous_loans": installments["SK_ID_PREV"].nunique(),
    "avg_rows_per_customer": (
        len(installments)
        / installments["SK_ID_CURR"].nunique()
    )
})

installments_summary

rows                     1.360540e+07
unique_customers         3.395870e+05
unique_previous_loans    9.977520e+05
avg_rows_per_customer    4.006455e+01
dtype: float64

In [155]:
installments[
    [
        "DAYS_INSTALMENT",
        "DAYS_ENTRY_PAYMENT",
        "AMT_INSTALMENT",
        "AMT_PAYMENT"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
DAYS_INSTALMENT,13605401.0,-1042.269992,800.946284,-2922.0,-1654.000,-818.000,-361.000,-1.000
DAYS_ENTRY_PAYMENT,13602496.0,-1051.113684,800.585883,-4921.0,-1662.000,-827.000,-370.000,-1.000
AMT_INSTALMENT,13605401.0,17050.906989,50570.254429,0.0,4226.085,8884.080,16710.210,3771487.845
AMT_PAYMENT,13602496.0,17238.223250,54735.783981,0.0,3398.265,8125.515,16108.425,3771487.845


In [156]:
installment_base_features = (
    installments
    .groupby(
        "SK_ID_CURR",
        sort=False
    )
    .agg(
        INST_PAYMENT_RECORD_COUNT=(
            "SK_ID_PREV",
            "count"
        ),
        INST_PREVIOUS_LOAN_COUNT=(
            "SK_ID_PREV",
            "nunique"
        ),

        INST_AMT_INSTALMENT_MEAN=(
            "AMT_INSTALMENT",
            "mean"
        ),
        INST_AMT_INSTALMENT_SUM=(
            "AMT_INSTALMENT",
            "sum"
        ),

        INST_AMT_PAYMENT_MEAN=(
            "AMT_PAYMENT",
            "mean"
        ),
        INST_AMT_PAYMENT_SUM=(
            "AMT_PAYMENT",
            "sum"
        ),

        INST_VALID_PAYMENT_COUNT=(
            "AMT_PAYMENT",
            "count"
        ),

        INST_DAYS_INSTALMENT_MEAN=(
            "DAYS_INSTALMENT",
            "mean"
        ),
        INST_DAYS_INSTALMENT_MAX=(
            "DAYS_INSTALMENT",
            "max"
        ),

        INST_DAYS_PAYMENT_MEAN=(
            "DAYS_ENTRY_PAYMENT",
            "mean"
        ),
        INST_DAYS_PAYMENT_MAX=(
            "DAYS_ENTRY_PAYMENT",
            "max"
        ),

        INST_VERSION_MAX=(
            "NUM_INSTALMENT_VERSION",
            "max"
        ),
        INST_NUMBER_MAX=(
            "NUM_INSTALMENT_NUMBER",
            "max"
        )
    )
    .reset_index()
)

print(
    f"Rows: {installment_base_features.shape[0]:,}"
)

print(
    f"Columns: {installment_base_features.shape[1]}"
)

installment_base_features.head()

Rows: 339,587
Columns: 14


,SK_ID_CURR,INST_PAYMENT_RECORD_COUNT,INST_PREVIOUS_LOAN_COUNT,INST_AMT_INSTALMENT_MEAN,INST_AMT_INSTALMENT_SUM,INST_AMT_PAYMENT_MEAN,INST_AMT_PAYMENT_SUM,INST_VALID_PAYMENT_COUNT,INST_DAYS_INSTALMENT_MEAN,INST_DAYS_INSTALMENT_MAX,INST_DAYS_PAYMENT_MEAN,INST_DAYS_PAYMENT_MAX,INST_VERSION_MAX,INST_NUMBER_MAX
0,161674,101,9,12600.013812,1272601.395,12600.013812,1272601.395,101,-1026.643564,-60.0,-1037.544554,-62.0,2.0,36
1,151639,158,5,10027.751582,1584384.750,9240.438418,1459989.270,158,-1327.765823,-13.0,-1330.822785,-13.0,2.0,104
2,193053,3,1,11483.070000,34449.210,11483.070000,34449.210,3,-40.333333,-14.0,-35.000000,-21.0,3.0,2
3,199697,27,2,20401.190000,550832.130,14910.156667,402574.230,27,-1117.037037,-574.0,-1116.370370,-581.0,3.0,102
4,167756,30,3,4114.450500,123433.515,3492.061500,104761.845,30,-1190.600000,-747.0,-1198.400000,-771.0,2.0,11


In [157]:
installments["__DPD"] = (
    installments["DAYS_ENTRY_PAYMENT"]
    - installments["DAYS_INSTALMENT"]
).clip(lower=0)

installments["__LATE"] = (
    installments["__DPD"] > 0
).astype("int8")

installments["__LATE_7"] = (
    installments["__DPD"] > 7
).astype("int8")

installments["__LATE_30"] = (
    installments["__DPD"] > 30
).astype("int8")

In [158]:
installment_dpd_features = (
    installments
    .groupby(
        "SK_ID_CURR",
        sort=False
    )
    .agg(
        INST_DPD_VALID_COUNT=(
            "__DPD",
            "count"
        ),

        INST_DPD_MEAN=(
            "__DPD",
            "mean"
        ),
        INST_DPD_MAX=(
            "__DPD",
            "max"
        ),
        INST_DPD_SUM=(
            "__DPD",
            "sum"
        ),

        INST_LATE_PAYMENT_COUNT=(
            "__LATE",
            "sum"
        ),
        INST_LATE_7_COUNT=(
            "__LATE_7",
            "sum"
        ),
        INST_LATE_30_COUNT=(
            "__LATE_30",
            "sum"
        )
    )
    .reset_index()
)

installment_dpd_features.head()

,SK_ID_CURR,INST_DPD_VALID_COUNT,INST_DPD_MEAN,INST_DPD_MAX,INST_DPD_SUM,INST_LATE_PAYMENT_COUNT,INST_LATE_7_COUNT,INST_LATE_30_COUNT
0,161674,101,0.000000,0.0,0.0,0,0,0
1,151639,158,0.202532,9.0,32.0,11,1,0
2,193053,3,7.666667,23.0,23.0,1,1,0
3,199697,27,7.925926,74.0,214.0,11,9,2
4,167756,30,2.100000,17.0,63.0,9,3,0


In [159]:
installments.drop(
    columns=[
        "__DPD",
        "__LATE",
        "__LATE_7",
        "__LATE_30"
    ],
    inplace=True
)

gc.collect()

print("Temporary DPD columns removed.")

Temporary DPD columns removed.


In [160]:
installments["__SHORTFALL"] = (
    installments["AMT_INSTALMENT"]
    - installments["AMT_PAYMENT"]
).clip(lower=0)

installments["__UNDERPAID"] = (
    installments["__SHORTFALL"] > 0
).astype("int8")

In [161]:
installment_payment_features = (
    installments
    .groupby(
        "SK_ID_CURR",
        sort=False
    )
    .agg(
        INST_SHORTFALL_MEAN=(
            "__SHORTFALL",
            "mean"
        ),
        INST_SHORTFALL_MAX=(
            "__SHORTFALL",
            "max"
        ),
        INST_SHORTFALL_SUM=(
            "__SHORTFALL",
            "sum"
        ),
        INST_UNDERPAID_COUNT=(
            "__UNDERPAID",
            "sum"
        )
    )
    .reset_index()
)

installment_payment_features.head()

,SK_ID_CURR,INST_SHORTFALL_MEAN,INST_SHORTFALL_MAX,INST_SHORTFALL_SUM,INST_UNDERPAID_COUNT
0,161674,0.000000,0.000,0.00,0
1,151639,787.313165,26067.465,124395.48,15
2,193053,0.000000,0.000,0.00,0
3,199697,5491.033333,21174.300,148257.90,14
4,167756,622.389000,2389.680,18671.67,16


In [162]:
installments.drop(
    columns=[
        "__SHORTFALL",
        "__UNDERPAID"
    ],
    inplace=True
)

gc.collect()

print("Temporary payment columns removed.")

Temporary payment columns removed.


In [163]:
installment_features = (
    installment_base_features
    .merge(
        installment_dpd_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
    .merge(
        installment_payment_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
)

In [164]:
installment_features["INST_LATE_PAYMENT_RATE"] = (
    installment_features["INST_LATE_PAYMENT_COUNT"]
    / installment_features["INST_DPD_VALID_COUNT"]
    .replace(0, np.nan)
)

installment_features["INST_LATE_7_RATE"] = (
    installment_features["INST_LATE_7_COUNT"]
    / installment_features["INST_DPD_VALID_COUNT"]
    .replace(0, np.nan)
)

installment_features["INST_LATE_30_RATE"] = (
    installment_features["INST_LATE_30_COUNT"]
    / installment_features["INST_DPD_VALID_COUNT"]
    .replace(0, np.nan)
)

installment_features["INST_UNDERPAID_RATE"] = (
    installment_features["INST_UNDERPAID_COUNT"]
    / installment_features["INST_VALID_PAYMENT_COUNT"]
    .replace(0, np.nan)
)

installment_features["INST_PAYMENT_TO_INSTALMENT_RATIO"] = (
    installment_features["INST_AMT_PAYMENT_SUM"]
    / installment_features["INST_AMT_INSTALMENT_SUM"]
    .replace(0, np.nan)
)

In [165]:
print(
    f"Rows: {installment_features.shape[0]:,}"
)

print(
    f"Columns: {installment_features.shape[1]}"
)

print(
    "Unique customers:",
    installment_features["SK_ID_CURR"].nunique()
)

print(
    "SK_ID_CURR unique:",
    installment_features["SK_ID_CURR"].is_unique
)

installment_features[
    [
        "SK_ID_CURR",
        "INST_PAYMENT_RECORD_COUNT",
        "INST_LATE_PAYMENT_RATE",
        "INST_LATE_7_RATE",
        "INST_LATE_30_RATE",
        "INST_UNDERPAID_RATE",
        "INST_PAYMENT_TO_INSTALMENT_RATIO"
    ]
].head(10)

Rows: 339,587
Columns: 30
Unique customers: 339587
SK_ID_CURR unique: True


,SK_ID_CURR,INST_PAYMENT_RECORD_COUNT,INST_LATE_PAYMENT_RATE,INST_LATE_7_RATE,INST_LATE_30_RATE,INST_UNDERPAID_RATE,INST_PAYMENT_TO_INSTALMENT_RATIO
0,161674,101,0.000000,0.000000,0.000000,0.000000,1.000000
1,151639,158,0.069620,0.006329,0.000000,0.094937,0.921487
2,193053,3,0.333333,0.333333,0.000000,0.000000,1.000000
3,199697,27,0.407407,0.333333,0.074074,0.518519,0.730847
4,167756,30,0.300000,0.100000,0.000000,0.533333,0.848731
5,164489,15,0.000000,0.000000,0.000000,0.000000,1.000000
6,184693,93,0.129032,0.000000,0.000000,0.000000,1.034213
7,111420,43,0.069767,0.069767,0.000000,0.139535,0.957102
8,112102,37,0.027027,0.027027,0.000000,0.000000,1.000000
9,109741,36,0.000000,0.000000,0.000000,0.000000,1.000000


In [166]:
installment_features_path = (
    INTERIM_DIR / "installments_customer_features.csv"
)

installment_features.to_csv(
    installment_features_path,
    index=False
)

print(f"Saved to: {installment_features_path}")

Saved to: c:\Projects\creditlens-ai\data\interim\installments_customer_features.csv


In [167]:
del installments
del installment_base_features
del installment_dpd_features
del installment_payment_features

gc.collect()

print("Raw installment objects removed from memory.")

Raw installment objects removed from memory.


In [168]:
application_train = pd.read_csv(
    DATA_DIR / "application_train.csv",
    low_memory=False
)

print(f"Application shape: {application_train.shape}")

Application shape: (307511, 122)


In [169]:
installment_coverage = (
    application_train[["SK_ID_CURR"]]
    .merge(
        installment_features[["SK_ID_CURR"]],
        on="SK_ID_CURR",
        how="left",
        indicator=True,
        validate="one_to_one"
    )
)

installment_coverage["_merge"].value_counts()

_merge
both          291643
left_only      15868
right_only         0
Name: count, dtype: int64

In [170]:
installment_matched = (
    installment_coverage["_merge"] == "both"
).sum()

installment_total = len(installment_coverage)

print(
    f"Customers with installment history: "
    f"{installment_matched:,}"
)

print(
    f"Coverage: "
    f"{installment_matched / installment_total * 100:.2f}%"
)

Customers with installment history: 291,643
Coverage: 94.84%


In [171]:
train_full_relational = (
    application_train
    .merge(
        bureau_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
    .merge(
        previous_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
    .merge(
        installment_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
)

print(
    f"Shape before history flags: "
    f"{train_full_relational.shape}"
)

Shape before history flags: (307511, 193)


In [172]:
train_full_relational["BUREAU_HAS_HISTORY"] = (
    train_full_relational["BUREAU_LOAN_COUNT"]
    .notna()
    .astype("int8")
)

train_full_relational["PREV_HAS_HISTORY"] = (
    train_full_relational["PREV_APPLICATION_COUNT"]
    .notna()
    .astype("int8")
)

train_full_relational["INST_HAS_HISTORY"] = (
    train_full_relational["INST_PAYMENT_RECORD_COUNT"]
    .notna()
    .astype("int8")
)

C:\Users\Vıctus\AppData\Local\Temp\ipykernel_23348\873887218.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_full_relational["BUREAU_HAS_HISTORY"] = (
C:\Users\Vıctus\AppData\Local\Temp\ipykernel_23348\873887218.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_full_relational["PREV_HAS_HISTORY"] = (
C:\Users\Vıctus\AppData\Local\Temp\ipykernel_23348\873887218.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance. 

In [173]:
bureau_count_columns = [
    "BUREAU_LOAN_COUNT",
    "BUREAU_STATUS_Active",
    "BUREAU_STATUS_Bad debt",
    "BUREAU_STATUS_Closed",
    "BUREAU_STATUS_Sold"
]

previous_count_columns = [
    "PREV_APPLICATION_COUNT",
    "PREV_STATUS_Approved",
    "PREV_STATUS_Canceled",
    "PREV_STATUS_Refused",
    "PREV_STATUS_Unused offer"
]

installment_count_columns = [
    "INST_PAYMENT_RECORD_COUNT",
    "INST_PREVIOUS_LOAN_COUNT",
    "INST_VALID_PAYMENT_COUNT",
    "INST_DPD_VALID_COUNT",
    "INST_LATE_PAYMENT_COUNT",
    "INST_LATE_7_COUNT",
    "INST_LATE_30_COUNT",
    "INST_UNDERPAID_COUNT"
]

count_columns = (
    bureau_count_columns
    + previous_count_columns
    + installment_count_columns
)

train_full_relational[count_columns] = (
    train_full_relational[count_columns]
    .fillna(0)
)

In [174]:
train_full_relational = (
    train_full_relational.copy()
)

print(
    f"Final relational shape: "
    f"{train_full_relational.shape}"
)

Final relational shape: (307511, 196)


In [175]:
print(
    "Duplicate customers:",
    train_full_relational["SK_ID_CURR"]
    .duplicated()
    .sum()
)

print(
    "Target missing:",
    train_full_relational["TARGET"]
    .isna()
    .sum()
)

print(
    "Bureau history rate:",
    f"{train_full_relational['BUREAU_HAS_HISTORY'].mean() * 100:.2f}%"
)

print(
    "Previous history rate:",
    f"{train_full_relational['PREV_HAS_HISTORY'].mean() * 100:.2f}%"
)

print(
    "Installment history rate:",
    f"{train_full_relational['INST_HAS_HISTORY'].mean() * 100:.2f}%"
)

Duplicate customers: 0
Target missing: 0
Bureau history rate: 85.69%
Previous history rate: 94.65%
Installment history rate: 94.84%


In [176]:
y_full = train_full_relational["TARGET"].copy()

X_full = (
    train_full_relational
    .drop(
        columns=[
            "TARGET",
            "SK_ID_CURR"
        ]
    )
    .copy()
)

X_full["DAYS_EMPLOYED_ANOMALY"] = (
    X_full["DAYS_EMPLOYED"] == 365243
).astype("int8")

X_full.loc[
    X_full["DAYS_EMPLOYED"] == 365243,
    "DAYS_EMPLOYED"
] = np.nan

print(f"X shape: {X_full.shape}")
print(f"y shape: {y_full.shape}")

X shape: (307511, 195)
y shape: (307511,)


In [177]:
X_train_full, X_val_full, y_train_full, y_val_full = train_test_split(
    X_full,
    y_full,
    test_size=0.20,
    random_state=42,
    stratify=y_full
)

print(f"Train shape: {X_train_full.shape}")
print(f"Validation shape: {X_val_full.shape}")

print(
    f"Train positive rate: "
    f"{y_train_full.mean() * 100:.2f}%"
)

print(
    f"Validation positive rate: "
    f"{y_val_full.mean() * 100:.2f}%"
)

Train shape: (246008, 195)
Validation shape: (61503, 195)
Train positive rate: 8.07%
Validation positive rate: 8.07%


In [178]:
numeric_columns_full = (
    X_train_full
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

categorical_columns_full = (
    X_train_full
    .select_dtypes(exclude=["number"])
    .columns
    .tolist()
)

print(
    f"Numeric features: "
    f"{len(numeric_columns_full)}"
)

print(
    f"Categorical features: "
    f"{len(categorical_columns_full)}"
)

print(
    f"Total features: "
    f"{len(numeric_columns_full) + len(categorical_columns_full)}"
)

Numeric features: 179
Categorical features: 16
Total features: 195


In [179]:
numeric_pipeline_full = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline_full = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

preprocessor_full = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline_full,
            numeric_columns_full
        ),
        (
            "categorical",
            categorical_pipeline_full,
            categorical_columns_full
        )
    ],
    sparse_threshold=1.0
)

preprocessor_full

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",1.0
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fe

In [180]:
full_relational_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_full
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                solver="saga",
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

full_relational_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",1.0
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be exclude

In [181]:
import time

start_time = time.time()

full_relational_model.fit(
    X_train_full,
    y_train_full
)

full_training_time = (
    time.time() - start_time
)

full_classifier = (
    full_relational_model
    .named_steps["classifier"]
)

print(
    f"Training time: "
    f"{full_training_time:.2f} seconds"
)

print(
    f"Iterations used: "
    f"{full_classifier.n_iter_[0]}"
)

Training time: 416.80 seconds
Iterations used: 1000


c:\Projects\creditlens-ai\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [182]:
full_probabilities = (
    full_relational_model
    .predict_proba(X_val_full)[:, 1]
)

full_predictions = (
    full_relational_model
    .predict(X_val_full)
)

In [183]:
full_metrics = {
    "accuracy": accuracy_score(
        y_val_full,
        full_predictions
    ),

    "precision": precision_score(
        y_val_full,
        full_predictions,
        zero_division=0
    ),

    "recall": recall_score(
        y_val_full,
        full_predictions,
        zero_division=0
    ),

    "f1": f1_score(
        y_val_full,
        full_predictions,
        zero_division=0
    ),

    "roc_auc": roc_auc_score(
        y_val_full,
        full_probabilities
    ),

    "pr_auc": average_precision_score(
        y_val_full,
        full_probabilities
    )
}

pd.Series(full_metrics)

accuracy     0.702649
precision    0.170272
recall       0.692850
f1           0.273363
roc_auc      0.765474
pr_auc       0.247268
dtype: float64

In [184]:
full_confusion_matrix = confusion_matrix(
    y_val_full,
    full_predictions
)

full_confusion_matrix

array([[39775, 16763],
       [ 1525,  3440]])

In [185]:
full_relational_model.set_params(
    classifier__max_iter=2000,
    classifier__warm_start=True
)

start_time = time.time()

full_relational_model.fit(
    X_train_full,
    y_train_full
)

full_converged_training_time = time.time() - start_time

full_classifier = (
    full_relational_model
    .named_steps["classifier"]
)

print(
    f"Training time: "
    f"{full_converged_training_time:.2f} seconds"
)

print(
    f"Iterations used: "
    f"{full_classifier.n_iter_[0]}"
)

Training time: 797.02 seconds
Iterations used: 2000


c:\Projects\creditlens-ai\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [186]:
full_converged_probabilities = (
    full_relational_model
    .predict_proba(X_val_full)[:, 1]
)

full_converged_predictions = (
    full_relational_model
    .predict(X_val_full)
)

full_converged_metrics = {
    "accuracy": accuracy_score(
        y_val_full,
        full_converged_predictions
    ),

    "precision": precision_score(
        y_val_full,
        full_converged_predictions,
        zero_division=0
    ),

    "recall": recall_score(
        y_val_full,
        full_converged_predictions,
        zero_division=0
    ),

    "f1": f1_score(
        y_val_full,
        full_converged_predictions,
        zero_division=0
    ),

    "roc_auc": roc_auc_score(
        y_val_full,
        full_converged_probabilities
    ),

    "pr_auc": average_precision_score(
        y_val_full,
        full_converged_probabilities
    )
}

pd.Series(full_converged_metrics)

accuracy     0.702811
precision    0.170258
recall       0.692246
f1           0.273298
roc_auc      0.765289
pr_auc       0.246884
dtype: float64

In [187]:
full_converged_confusion_matrix = confusion_matrix(
    y_val_full,
    full_converged_predictions
)

full_converged_confusion_matrix

array([[39788, 16750],
       [ 1528,  3437]])

In [188]:
final_feature_comparison = pd.DataFrame(
    {
        "Application_Only": {
            "accuracy": 0.689300,
            "precision": 0.161368,
            "recall": 0.678751,
            "f1": 0.260745,
            "roc_auc": 0.748826,
            "pr_auc": 0.228761
        },

        "Application_Plus_Bureau": {
            "accuracy": 0.691348,
            "precision": 0.163353,
            "recall": 0.684995,
            "f1": 0.263797,
            "roc_auc": 0.753412,
            "pr_auc": 0.235836
        },

        "Application_Plus_Bureau_Plus_Previous": {
            "accuracy": 0.696307,
            "precision": 0.166982,
            "recall": 0.692447,
            "f1": 0.269077,
            "roc_auc": 0.760525,
            "pr_auc": 0.243105
        },

        "Application_Plus_Bureau_Plus_Previous_Plus_Installments": {
            "accuracy": 0.702649,
            "precision": 0.170272,
            "recall": 0.692850,
            "f1": 0.273363,
            "roc_auc": 0.765474,
            "pr_auc": 0.247268
        }
    }
).T

final_feature_comparison

,accuracy,precision,recall,f1,roc_auc,pr_auc
Application_Only,0.689300,0.161368,0.678751,0.260745,0.748826,0.228761
Application_Plus_Bureau,0.691348,0.163353,0.684995,0.263797,0.753412,0.235836
Application_Plus_Bureau_Plus_Previous,0.696307,0.166982,0.692447,0.269077,0.760525,0.243105
Application_Plus_Bureau_Plus_Previous_Plus_Installments,0.702649,0.170272,0.692850,0.273363,0.765474,0.247268


In [189]:
feature_incremental_gain = (
    final_feature_comparison[
        ["roc_auc", "pr_auc", "f1"]
    ]
    .diff()
)

feature_incremental_gain

,roc_auc,pr_auc,f1
Application_Only,NaN,NaN,NaN
Application_Plus_Bureau,0.004586,0.007075,0.003052
Application_Plus_Bureau_Plus_Previous,0.007113,0.007269,0.005280
Application_Plus_Bureau_Plus_Previous_Plus_Installments,0.004949,0.004163,0.004286


In [190]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

final_comparison_path = (
    REPORTS_DIR / "final_relational_feature_comparison.csv"
)

incremental_gain_path = (
    REPORTS_DIR / "relational_incremental_gain.csv"
)

final_feature_comparison.to_csv(
    final_comparison_path,
    index=True
)

feature_incremental_gain.to_csv(
    incremental_gain_path,
    index=True
)

print(f"Saved: {final_comparison_path}")
print(f"Saved: {incremental_gain_path}")

Saved: c:\Projects\creditlens-ai\reports\final_relational_feature_comparison.csv
Saved: c:\Projects\creditlens-ai\reports\relational_incremental_gain.csv


## Final Relational Feature Engineering Result

Three relational data sources were incrementally added to the application-level dataset:

1. Bureau credit history
2. Previous application history
3. Installment payment history

Validation performance improved consistently as additional relational information was introduced.

| Feature Set | ROC-AUC | PR-AUC | F1 |
|---|---:|---:|---:|
| Application Only | 0.7488 | 0.2288 | 0.2607 |
| + Bureau | 0.7534 | 0.2358 | 0.2638 |
| + Previous Applications | 0.7605 | 0.2431 | 0.2691 |
| + Installment Payments | 0.7655 | 0.2473 | 0.2734 |

Overall improvement relative to the application-only baseline:

- ROC-AUC: +0.0166
- PR-AUC: +0.0185
- F1: +0.0126

Previous application history produced the largest single incremental improvement, while bureau and installment-payment information also contributed additional predictive signal.

The full Logistic Regression model reached the iteration limit during optimization. Increasing the iteration limit produced negligible metric changes, indicating that further Logistic Regression iterations were unlikely to provide meaningful performance gains.

The relational feature set will therefore be retained for subsequent modeling experiments with stronger nonlinear models.